## 1. Confirm all expected tables actually exist

In [0]:
display(spark.sql("SHOW TABLES IN workspace.cryptolens"))

# You should see: bronze_klines, bronze_exchange_info (only if you've 
# run that cell), bronze_quarantine (only if any files had bad rows), 
# silver_klines, dim_symbol (only if exchangeInfo has been ingested), 
# # pipeline_execution_logs.

## 2. Check row counts and distinct symbols at each layer

In [0]:
print("bronze_klines row count:", spark.table("workspace.cryptolens.bronze_klines").count())
print("bronze_klines distinct symbols:", spark.table("workspace.cryptolens.bronze_klines").select("symbol").distinct().count())

print("silver_klines row count:", spark.table("workspace.cryptolens.silver_klines").count())
print("silver_klines distinct symbols:", spark.table("workspace.cryptolens.silver_klines").select("symbol").distinct().count())

# These two symbol counts should roughly match (Silver drops a few rows 
# via quality filters, so Silver's row count will be slightly lower than 
# Bronze's, but the symbol count should be the same or very close). If 
# Bronze shows 50 symbols but Silver shows far fewer, that tells you 
# some symbols are silently failing Silver's quality filters entirely.

## 3. Prove idempotency

In [0]:
before = spark.table("workspace.cryptolens.silver_klines").count()
# now re-run your Silver driver loop cell manually, then come back and run this:
after = spark.table("workspace.cryptolens.silver_klines").count()
print(f"Before: {before}, After: {after}, Match: {before == after}")

## 4. Spot-check the actual data looks sane, not just present

In [0]:
display(spark.table("workspace.cryptolens.silver_klines")
         .filter("symbol = 'BTCUSDT'")
         .orderBy("open_time")
         .limit(10))

## 5. Confirm the audit log actually recorded everything

In [0]:
display(spark.table("workspace.cryptolens.pipeline_execution_logs")
         .orderBy("start_time", ascending=False)
         .limit(20))

# Look for: a SUCCESS row per symbol file processed, status column has 
# no unexpected FAILURE rows, rows_inserted/rows_updated are non-zero,
#  no stray SKIPPED rows pointing at symbols you actually expected to 
# process.

In [0]:
# Quick health check: any failures at all, across the whole run history?
display(spark.table("workspace.cryptolens.pipeline_execution_logs").filter("status = 'FAILURE'"))

## 6. Check the quarantine table — did any files have bad data?

In [0]:
if spark.catalog.tableExists("workspace.cryptolens.bronze_quarantine"):
    display(spark.table("workspace.cryptolens.bronze_quarantine").limit(10))
    print("Quarantined row count:", spark.table("workspace.cryptolens.bronze_quarantine").count())
else:
    print("No quarantine table — no malformed rows were ever encountered.")

## 7. Verify the quality-filter rules are actually filtering (not silently passing everything)

In [0]:
# This counts how many Bronze rows got dropped during the Bronze -> Silver quality checks
bronze_count = spark.table("workspace.cryptolens.bronze_klines").filter("load_type='full'").count()
silver_count = spark.table("workspace.cryptolens.silver_klines").count()
print(f"Bronze: {bronze_count}, Silver: {silver_count}, Dropped by quality rules: {bronze_count - silver_count}")

# A small number dropped is expected/healthy (confirms the filters 
# work). Zero dropped across ~2 million rows would actually be slightly 
# suspicious — worth a second look rather than assuming it's fine.

# Run through these in order — if any step surprises you (counts don't 
# match expectations, a FAILURE row shows up, symbol counts diverge 
# between layers), paste what you see and I'll help interpret it.